In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q docling-core transformers accelerate pillow
import torch
print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.6/305.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
GPU: T

In [3]:
import glob, pandas as pd
from pathlib import Path

RUN = pd.read_csv(glob.glob("/kaggle/input/**/run_list.csv", recursive=True)[0])
IMG = {}
for p in glob.glob("/kaggle/input/**/batch1-*.jpg", recursive=True):
    if "batch_3" in p:
        continue
    IMG.setdefault(Path(p).stem, p)
RUN["path"] = RUN["file_name"].map(IMG)
RUN = RUN.dropna(subset=["path"]).reset_index(drop=True)
print(f"실행 목록 {len(RUN):,}건 / 이미지 {len(IMG):,}장")

실행 목록 1,413건 / 이미지 1,489장


In [4]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL  = "ibm-granite/granite-docling-258M"
PROMPT = "Convert this page to OTSL."

proc  = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL, dtype=torch.bfloat16, attn_implementation="sdpa").to("cuda").eval()
print("로드 완료 |", next(model.parameters()).dtype, "|", PROMPT)

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/588 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/812 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/515M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/470 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/169 [00:00<?, ?B/s]

로드 완료 | torch.bfloat16 | Convert this page to OTSL.


In [5]:
from PIL import Image
import torch

def parse(path, max_tokens=3000):
    img = Image.open(path).convert("RGB")
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": PROMPT}]}]
    inp = proc.apply_chat_template(msgs, add_generation_prompt=True,
                                   tokenize=True, return_dict=True,
                                   return_tensors="pt").to("cuda")
    with torch.no_grad():
        o = model.generate(**inp, max_new_tokens=max_tokens,
                           do_sample=False, use_cache=True)
    return proc.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=False)

In [6]:
import json, time
from pathlib import Path

OUT, END = Path("/kaggle/working/granite_raw.jsonl"), 150
t0, n_ok, n_err = time.time(), 0, 0

with OUT.open("w", encoding="utf-8") as f:
    for _, row in RUN.iloc[:END].iterrows():
        t = time.time()
        try:
            rec = {"file_name": row["file_name"], "seq": int(row["seq"]), "ok": True,
                   "output": parse(row["path"]), "elapsed": round(time.time()-t, 2)}
            n_ok += 1
        except Exception as e:
            rec = {"file_name": row["file_name"], "seq": int(row["seq"]), "ok": False,
                   "output": "", "error": str(e)[:300], "elapsed": round(time.time()-t, 2)}
            n_err += 1
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        n = n_ok + n_err
        if n % 25 == 0:
            el = time.time() - t0
            print(f"{n:>4}/{END}  경과 {el/60:.0f}분  남은 {(END-n)*el/n/60:.0f}분  실패 {n_err}")

print(f"\n완료 성공 {n_ok} 실패 {n_err} {(time.time()-t0)/60:.0f}분")
print(f"파일 {OUT.stat().st_size/1e6:.1f}MB / {sum(1 for _ in OUT.open(encoding='utf-8'))}건")

  25/150  경과 13분  남은 63분  실패 0
  50/150  경과 24분  남은 47분  실패 0
  75/150  경과 34분  남은 34분  실패 0
 100/150  경과 45분  남은 22분  실패 0
 125/150  경과 56분  남은 11분  실패 0
 150/150  경과 66분  남은 0분  실패 0

완료 성공 150 실패 0 66분
파일 0.3MB / 150건
